# Few-Shot Prompt Templates

The `few_shot.py` module defines string-based and chat-based prompt templates that insert predefined or dynamically selected examples into a prompt.

# Shared Few-Shot Prompt Behaviour

`FewShotPromptTemplate` and `FewShotChatMessagePromptTemplate` inherit common example-management behaviour from the internal `_FewShotPromptTemplateMixin` class. The mixin requires exactly one of `examples` or `example_selector` to be provided.

## Inherited Fields

1. `examples`:`list[dict[str, Any]] | None`:= Stores a fixed list of example dictionaries used when formatting the prompt. Its default value is `None`.

2. `example_selector`:`BaseExampleSelector | None`:= Stores an optional selector that dynamically chooses examples based on the formatting inputs. Its default value is `None`.

## Inherited Configuration

1. `model_config`:`ConfigDict`:= Allows arbitrary Python types and rejects undeclared fields.

   **Syntax**

   ```python
   model_config = ConfigDict(
       arbitrary_types_allowed=True, # Allow arbitrary Python types
       extra="forbid" # Reject undeclared fields
   )
   ```

## Inherited Validators

1. `check_examples_and_selector`:= Validates that exactly one of `examples` or `example_selector` is provided.

   This validator runs automatically while the prompt template is being created.

   **Syntax**

   ```python
   @classmethod
   check_examples_and_selector(
       cls, # Prompt-template class
       values: dict[str, Any] # Values supplied while creating the prompt template
   ) -> Any
   ```

In [1]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from pydantic import ValidationError

examples = [
    {"word": "happy", "opposite": "sad"},
    {"word": "hot", "opposite": "cold"}
]

example_prompt = PromptTemplate.from_template(
    "Word: {word}\nOpposite: {opposite}"
)

# Valid: only fixed examples are provided
prompt = FewShotPromptTemplate(
    examples=examples,
    example_selector=None,
    example_prompt=example_prompt,
    prefix="Provide the opposite of each word.",
    suffix="Word: {input_word}\nOpposite:",
    input_variables=["input_word"]
)

print(prompt.format(input_word="fast"))

# Invalid: neither examples nor example_selector is provided
try:
    FewShotPromptTemplate(
        example_prompt=example_prompt,
        suffix="Word: {input_word}\nOpposite:"
    )
except ValidationError as error:
    print("\nValidation error:")
    print(error)

# Invalid: an undeclared field is rejected because extra="forbid"
try:
    FewShotPromptTemplate(
        examples=examples,
        example_prompt=example_prompt,
        suffix="Word: {input_word}\nOpposite:",
        unknown_field="value"
    )
except ValidationError as error:
    print("\nUndeclared-field error:")
    print(error)

Provide the opposite of each word.

Word: happy
Opposite: sad

Word: hot
Opposite: cold

Word: fast
Opposite:

Validation error:
1 validation error for FewShotPromptTemplate
  Value error, One of 'examples' and 'example_selector' should be provided [type=value_error, input_value={'example_prompt': Prompt...': ['opposite', 'word']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

Undeclared-field error:
1 validation error for FewShotPromptTemplate
unknown_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='value', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


# FewShotPromptTemplate: `_FewShotPromptTemplateMixin`, `StringPromptTemplate`

`FewShotPromptTemplate` creates a string prompt by formatting multiple examples with an `example_prompt`, joining them with an optional prefix and suffix, and substituting the final input variables.

**Syntax**

```python
FewShotPromptTemplate(
    self, # Few-shot prompt template instance
    **kwargs: Any # Fields used to initialise the prompt template
)
```

## Fields

1. `validate_template`:`bool`:= Determines whether the prefix, suffix, partial variables, and declared input variables are validated for consistency. Its default value is `False`.

2. `example_prompt`:`PromptTemplate`:= Stores the prompt template used to format each individual example.

3. `suffix`:`str`:= Stores the prompt-template string placed after the formatted examples.

4. `example_separator`:`str`:= Stores the separator used to join the prefix, formatted examples, and suffix. Its default value is `"\n\n"`.

5. `prefix`:`str`:= Stores the prompt-template string placed before the formatted examples. Its default value is `""`.

6. `template_format`:`Literal["f-string", "jinja2"]`:= Specifies the formatting syntax used by the prefix and suffix templates. Its default value is `"f-string"`.

## Configuration

1. `model_config`:`ConfigDict`:= Allows arbitrary Python types and rejects undeclared fields.

   **Syntax**

   ```python
   model_config = ConfigDict(
       arbitrary_types_allowed=True, # Allow arbitrary Python types
       extra="forbid" # Reject undeclared fields
   )
   ```

## Validators

1. `template_is_valid`:= Validates the declared input variables against the prefix, suffix, and partial variables when template validation is enabled.

   When validation is disabled, the required input variables are derived automatically from the prefix and suffix. This validator runs automatically after the prompt template is created.

   **Syntax**

   ```python
   template_is_valid(
       self # Prompt template instance to validate
   ) -> Self
   ```

## Methods

1. `is_lc_serializable`:= Returns `False` because this prompt-template class does not support LangChain serialization.

   **Syntax**

   ```python
   @classmethod
   is_lc_serializable(
       cls # FewShotPromptTemplate class
   ) -> bool
   ```

2. `__init__`:= Initialises the few-shot prompt template and infers `input_variables` from `example_prompt` when they are not supplied explicitly.

   **Syntax**

   ```python
   __init__(
       self, # Few-shot prompt template instance
       **kwargs: Any # Fields used to initialise the prompt template
   ) -> None
   ```

3. `format`:= Synchronously retrieves fixed examples or selects examples dynamically, formats each example, combines all prompt sections, and returns the final string.

   **Syntax**

   ```python
   format(
       self, # Few-shot prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

4. `aformat`:= Asynchronously retrieves fixed examples or selects examples dynamically, formats each example, combines all prompt sections, and returns the final string.

   **Syntax**

   ```python
   async aformat(
       self, # Few-shot prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

5. `save`:= Saves the prompt template to a file when fixed examples are used.

   Saving a prompt that uses an `example_selector` is not supported. This method is deprecated in favour of the LangChain load and dump utilities.

   **Syntax**

   ```python
   save(
       self, # Few-shot prompt template instance
       file_path: Path | str # Destination path for the serialized prompt
   ) -> None
   ```

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate # Import the required prompt classes

examples = [ # Define the fixed few-shot examples
    {"question": "2 + 2", "answer": "4"}, # Store the first example
    {"question": "3 + 5", "answer": "8"} # Store the second example
] # End the examples list

example_prompt = PromptTemplate.from_template( # Create the template used for every example
    "Question: {question}\nAnswer: {answer}" # Define the format of each example
) # Finish creating the example prompt

prompt = FewShotPromptTemplate( # Initialise the few-shot prompt template
    examples=examples, # Provide fixed examples instead of an example selector
    example_prompt=example_prompt, # Specify how each example is formatted
    prefix="Solve the following addition problems.", # Add text before the examples
    suffix="Question: {user_question}\nAnswer:", # Add the final question after the examples
    example_separator="\n\n", # Separate each prompt section with two new lines
    template_format="f-string", # Use f-string variable formatting
    input_variables=["user_question"], # Declare the variable required by the suffix
    validate_template=True # Validate the declared variables automatically
) # Finish creating the prompt

print(prompt.input_variables) # Display the validated input variables
print(prompt.is_lc_serializable()) # Display False because serialization is unsupported

sync_result = prompt.format(user_question="6 + 7") # Format the prompt synchronously
print(sync_result) # Display the synchronously formatted prompt

async_result = await prompt.aformat(user_question="8 + 9") # Format the prompt asynchronously in Jupyter
print(async_result) # Display the asynchronously formatted prompt

prompt.save("few_shot_prompt.json") # Save the fixed-example prompt using the deprecated save method

# FewShotChatMessagePromptTemplate: `BaseChatPromptTemplate`, `_FewShotPromptTemplateMixin`

`FewShotChatMessagePromptTemplate` creates chat messages from few-shot examples. Each selected or fixed example is formatted through a message prompt or chat prompt, and the resulting messages are combined into a single list.

## Fields

1. `input_variables`:`list[str]`:= Stores the names of variables passed to the example selector when examples are selected dynamically. Its default value is an empty list.

2. `example_prompt`:`BaseMessagePromptTemplate | BaseChatPromptTemplate`:= Stores the message prompt or chat prompt used to format each example.

## Configuration

1. `model_config`:`ConfigDict`:= Allows arbitrary Python types and rejects undeclared fields.

   **Syntax**

   ```python
   model_config = ConfigDict(
       arbitrary_types_allowed=True, # Allow arbitrary Python types
       extra="forbid" # Reject undeclared fields
   )
   ```

## Methods

1. `is_lc_serializable`:= Returns `False` because this prompt-template class does not support LangChain serialization.

   **Syntax**

   ```python
   @classmethod
   is_lc_serializable(
       cls # FewShotChatMessagePromptTemplate class
   ) -> bool
   ```

2. `format_messages`:= Synchronously retrieves fixed examples or selects examples dynamically, formats each example into messages, and returns the combined message list.

   **Syntax**

   ```python
   format_messages(
       self, # Few-shot chat prompt template instance
       **kwargs: Any # Variables used for example selection and message formatting
   ) -> list[BaseMessage]
   ```

3. `aformat_messages`:= Asynchronously retrieves fixed examples or selects examples dynamically, formats each example into messages, and returns the combined message list.

   **Syntax**

   ```python
   async aformat_messages(
       self, # Few-shot chat prompt template instance
       **kwargs: Any # Variables used for example selection and message formatting
   ) -> list[BaseMessage]
   ```

4. `format`:= Synchronously formats the few-shot examples as chat messages and converts the messages into a single string.

   **Syntax**

   ```python
   format(
       self, # Few-shot chat prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

5. `aformat`:= Asynchronously formats the few-shot examples as chat messages and converts the messages into a single string.

   **Syntax**

   ```python
   async aformat(
       self, # Few-shot chat prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

6. `pretty_repr`:= Declares a human-readable representation of the prompt template.

   This method is not implemented and raises `NotImplementedError`.

   **Syntax**

   ```python
   pretty_repr(
       self, # Few-shot chat prompt template instance
       html: bool = False # Whether to produce an HTML-formatted representation
   ) -> str
   ```


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate # Import the required prompt-template classes

examples = [ # Define fixed few-shot examples
    {"question": "2 + 2", "answer": "4"}, # Store the first example
    {"question": "3 + 5", "answer": "8"} # Store the second example
] # End the examples list

example_prompt = ChatPromptTemplate.from_messages([ # Create the chat prompt used to format each example
    ("human", "{question}"), # Convert each question into a human message
    ("ai", "{answer}") # Convert each answer into an AI message
]) # Finish creating the example prompt

prompt = FewShotChatMessagePromptTemplate( # Create the few-shot chat message template
    examples=examples, # Provide fixed examples
    example_prompt=example_prompt, # Specify how each example becomes chat messages
    input_variables=[] # No variables are required for selecting fixed examples
) # Finish creating the few-shot template

print(prompt.is_lc_serializable()) # Display False because LangChain serialization is unsupported

messages = prompt.format_messages() # Format the examples synchronously as messages
for message in messages: # Iterate through the formatted messages
    print(message.type, message.content) # Display each message type and content

async_messages = await prompt.aformat_messages() # Format the examples asynchronously in Jupyter
for message in async_messages: # Iterate through the asynchronously formatted messages
    print(message.type, message.content) # Display each message type and content

print(prompt.format()) # Format the messages synchronously as one string
print(await prompt.aformat()) # Format the messages asynchronously as one string

try: # Test the unimplemented representation method
    print(prompt.pretty_repr()) # Attempt to create a human-readable representation
except NotImplementedError as error: # Catch the expected error
    print(type(error).__name__) # Display the error type